# Kaggle held-out evaluation: full-data Laya vs DibaOne X1

Evaluate the completed best checkpoint on held-out ParsiNLU tasks and PersianQA.
Converters are strict and test-only; outputs contain hashed IDs, labels,
predictions and aggregate reports, not source text. Keep this Kaggle notebook
private because source licenses prohibit publishing converted benchmark data.

Enable Internet and a Kaggle GPU (T4 recommended). Add an HF_TOKEN Kaggle Secret
with read access to Farahani/kodoom. The model and data revisions are recorded.
Diba X1 is choice-only; PersianQA answerability is explicitly converted to a
two-choice task and tagged as such in each prediction.

In [ ]:
from pathlib import Path

CODE_BRANCH = 'codex/evaluate-parsinlu-persianqa'
MODEL_REPO = 'Farahani/kodoom'
MODEL_RUN_ID = 'laya-full-v1'
MODEL_REVISION = 'f315b5da129e2b1df55d971ddf2053c3cbfe5282'
DIBA_MODEL_ID = 'Dibachain/DibaOne-X1'
PARSINLU_TASKS = {
    'entailment': 'persiannlp/parsinlu_entailment',
    'paraphrase': 'persiannlp/parsinlu_paraphrase',
    'sentiment': 'persiannlp/parsinlu_sentiment',
    'multiple-choice-qa': 'persiannlp/parsinlu_mcq',
}
WORK = Path('/kaggle/working/kodoom-eval')
CHECKOUT = WORK / 'kodum'
OUTPUT = WORK / 'results'
WORK.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)

## Bootstrap exact code and runtime

Push the evaluation branch to GitHub before running this cell. The resolved
commit SHA is saved with the results.

In [ ]:
import hashlib, json, os, subprocess, sys
from pathlib import Path

if not CHECKOUT.exists():
    subprocess.run(['git', 'clone', '--branch', CODE_BRANCH, '--single-branch',
                    'https://github.com/Farahani1/kodum.git', str(CHECKOUT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e',
                f'{CHECKOUT}[train,eval]'], check=True)
# Editable-install .pth files are not reprocessed by an already running kernel.
import importlib
source_root = (CHECKOUT / 'src').resolve()
sys.path.insert(0, str(source_root))
importlib.invalidate_caches()
import kodoom
if Path(kodoom.__file__).resolve().parent != source_root / 'kodoom':
    raise RuntimeError('A different kodoom checkout is already imported; restart the kernel.')
print('kodoom imported from:', kodoom.__file__)
CODE_REVISION = subprocess.check_output(
    ['git', '-C', str(CHECKOUT), 'rev-parse', 'HEAD'], text=True).strip()
print('kodum code revision:', CODE_REVISION)

## Load token and immutable best checkpoint

Downloads only best inference weights and the complete-snapshot manifest.
Every downloaded best file is checked against its manifest SHA-256.

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, snapshot_download
from kodoom.package_import import file_sha
from kodoom.training import laya

HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
model_root = WORK / 'model-snapshot'
snapshot_download(
    repo_id=MODEL_REPO, repo_type='model', revision=MODEL_REVISION, token=HF_TOKEN,
    allow_patterns=[f'{MODEL_RUN_ID}/durable-manifest.json', f'{MODEL_RUN_ID}/best/**'],
    local_dir=str(model_root),
)
snapshot_dir = model_root / MODEL_RUN_ID
manifest = json.loads((snapshot_dir / 'durable-manifest.json').read_text('utf-8'))
best_dir = snapshot_dir / 'best'
for rel, entry in manifest['files'].items():
    if rel.startswith('best/'):
        path = (snapshot_dir / rel).resolve()
        if not path.is_relative_to(best_dir.resolve()) or file_sha(path) != entry['sha256']:
            raise RuntimeError(f'Best-checkpoint manifest/hash mismatch: {rel}')
model, tokenizer, model_cfg, common = laya.load(WORK / 'cache', artifact=best_dir)
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device).eval()
# Use the model card's published inference module directly from the same
# immutable revision as its weights; there is no `dibaone` PyPI package.
diba_revision = HfApi().model_info(DIBA_MODEL_ID).sha
diba_root = WORK / 'diba-x1-snapshot'
snapshot_download(
    repo_id=DIBA_MODEL_ID, revision=diba_revision,
    allow_patterns=['dibaone_x1.py', '*.json', '*.safetensors', 'stage1/**'],
    local_dir=str(diba_root),
)
sys.path.insert(0, str(diba_root))
from dibaone_x1 import DibaOneX1
diba = DibaOneX1.from_pretrained(str(diba_root), backend='torch', device=device)
print('best checkpoint:', MODEL_REVISION, 'Diba X1 revision:', diba_revision, 'device:', device)

## Load held-out sources and convert to the project schema

ParsiNLU loads the first available test, validation, or dev split. PersianQA is
cloned at its current Git commit and only explicitly named test/dev/validation
files are accepted. Conversion stops on unknown schemas or label maps.

In [ ]:
from kodoom.external_eval import load_parsinlu_task, load_persianqa_repository

records_by_benchmark = {}
dataset_revisions = {}
for task, repo_id in PARSINLU_TASKS.items():
    records, revision, source_split = load_parsinlu_task(repo_id, task)
    records_by_benchmark[f'parsinlu-{task}'] = records
    dataset_revisions[repo_id] = {
        'revision': revision, 'split': source_split, 'rows': len(records)}
    print(task, source_split, len(records), revision)
persianqa_records, persianqa_revision, persianqa_splits = load_persianqa_repository(
    'https://github.com/sajjjadayobi/PersianQA.git', WORK / 'PersianQA')
records_by_benchmark['persianqa'] = persianqa_records
dataset_revisions['sajjjadayobi/PersianQA'] = {
    'revision': persianqa_revision, 'split': persianqa_splits,
    'rows': len(persianqa_records)}
for records in records_by_benchmark.values():
    assert records and all(record.split == 'test' for record in records)
print('total held-out records:', sum(map(len, records_by_benchmark.values())))

## Predict, score, and preserve all results

Both models receive identical records and options. Diba failures remain in its
prediction file and reduce coverage. Shared scoring reports accuracy, NLL,
Brier, KL, calibration, latency and source-case-bootstrap intervals.

In [ ]:
from kodoom.evaluation import evaluate, format_table
from kodoom.metrics import paired_difference_ci
from kodoom.external_eval import predict_diba_x1, predict_laya
from kodoom.predictions import write_predictions

all_report = {
    'protocol': 'external-heldout-v1', 'model_revision': MODEL_REVISION,
    'diba_revision': diba_revision, 'datasets': dataset_revisions, 'benchmarks': {},
}
for benchmark, records in records_by_benchmark.items():
    laya_preds = predict_laya(
        records, model, tokenizer, common, model_cfg, device=device, microbatch=4)
    diba_preds = predict_diba_x1(records, diba, mode='best')
    write_predictions(OUTPUT / benchmark / 'laya-best.jsonl', laya_preds)
    write_predictions(OUTPUT / benchmark / 'dibaone-x1.jsonl', diba_preds)
    laya_map = {prediction.id: prediction for prediction in laya_preds}
    diba_map = {prediction.id: prediction for prediction in diba_preds}
    laya_eval, diba_eval = evaluate(records, laya_map), evaluate(records, diba_map)
    common_records = [
        record for record in records
        if record.id in laya_map and record.id in diba_map
        and not laya_map[record.id].error and not diba_map[record.id].error
    ]
    laya_common = evaluate(common_records, laya_map)
    diba_common = evaluate(common_records, diba_map)
    laya_correct = {item.record.id: float(item.metrics.correct) for item in laya_common.items}
    diba_correct = {item.record.id: float(item.metrics.correct) for item in diba_common.items}
    common_report = {
        'n': len(common_records),
        'accuracy_difference_laya_minus_diba_ci95': paired_difference_ci(
            [laya_correct[record.id] for record in common_records],
            [diba_correct[record.id] for record in common_records],
            [record.source_id for record in common_records], n_boot=2000, seed=17),
        'laya': evaluate(common_records, laya_map).summary(n_boot=2000, seed=17).to_dict(),
        'dibaone_x1': evaluate(common_records, diba_map).summary(n_boot=2000, seed=17).to_dict(),
    }
    all_report['benchmarks'][benchmark] = {
        'n_records': len(records),
        'laya': laya_eval.summary(n_boot=2000, seed=17).to_dict(),
        'dibaone_x1': diba_eval.summary(n_boot=2000, seed=17).to_dict(),
        'diba_coverage': len(diba_eval.items) / len(records),
        'diba_failed': len(diba_eval.failed),
        'common_items': common_report,
        'protocols': sorted({
            record.extra.get('protocol', 'native-choice') for record in records
        }),
    }
    print('\n' + benchmark)
    print(format_table({
        'Laya best': laya_eval.summary(), 'Diba X1': diba_eval.summary()
    }, 'model'))

(OUTPUT / 'report.json').write_text(
    json.dumps(all_report, ensure_ascii=False, indent=2), encoding='utf-8')
output_hashes = {
    str(path.relative_to(OUTPUT)): hashlib.sha256(path.read_bytes()).hexdigest()
    for path in OUTPUT.rglob('*')
    if path.is_file() and path.name != 'manifest.json'
}
manifest_out = {
    'code_revision': CODE_REVISION, 'model_repo': MODEL_REPO,
    'model_revision': MODEL_REVISION, 'model_run_id': MODEL_RUN_ID,
    'model_path': f'{MODEL_RUN_ID}/best', 'diba_model': DIBA_MODEL_ID,
    'diba_revision': diba_revision, 'datasets': dataset_revisions,
    'protocol': 'external-heldout-v1', 'output_sha256': output_hashes,
}
(OUTPUT / 'manifest.json').write_text(
    json.dumps(manifest_out, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved predictions, report.json, manifest.json under', OUTPUT)